In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

data_train = pd.read_csv("csv/data_train.csv", header=0, dtype={"video": str})
data_test  = pd.read_csv("csv/data_test.csv",  header=0, dtype={"video": str})

X_train_all = data_train.drop(columns=["video", "label"])
X_test_all  = data_test.drop(columns=["video", "label"])
y_train = data_train["label"]
y_test  = data_test["label"]

rf = RandomForestClassifier(n_estimators=100)

In [ ]:
ALL_DIV  = r"(js|r|t)"
ALL_BASE = r"(10|20|40|60)"
ALL_FREQ = r"[1-9]"
ALL_QUAL = r"(80|85|90|95|100)"

bases_regex = [
    rf"^{ALL_DIV}_10_{ALL_FREQ}_{ALL_QUAL}$",
    rf"^{ALL_DIV}_(10|20)_{ALL_FREQ}_{ALL_QUAL}$",
    rf"^{ALL_DIV}_(10|20|40)_{ALL_FREQ}_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
]

frequencies_regex = [
    rf"^{ALL_DIV}_{ALL_BASE}_1_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[12]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[123]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[1234]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[12345]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[123456]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[1234567]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_[12345678]_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
]

qualities_regex = [
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_80$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_(80|85)$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_(80|85|90)$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_(80|85|90|95)$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
]

divergences_regex = [
    rf"^js_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
    rf"^(js|r)_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
    rf"^{ALL_DIV}_{ALL_BASE}_{ALL_FREQ}_{ALL_QUAL}$",
]


In [ ]:
acc_bases = []
for regex in bases_regex:
    X_tr = X_train_all.filter(regex=regex)
    X_te = X_test_all.filter(regex=regex)
    rf.fit(X_tr, y_train)
    acc_bases.append(accuracy_score(y_test, rf.predict(X_te)))
print("Bases:", acc_bases)

plt.figure(figsize=(6, 4))
plt.plot([1, 2, 3, 4], acc_bases, marker="o")
plt.xlabel("Numero de bases")
plt.ylabel("Exactitud")
plt.grid(True)
plt.tight_layout()
plt.savefig("accuracy_bases.jpg")
plt.show()

In [ ]:
acc_freqs = []
for regex in frequencies_regex:
    X_tr = X_train_all.filter(regex=regex)
    X_te = X_test_all.filter(regex=regex)
    rf.fit(X_tr, y_train)
    acc_freqs.append(accuracy_score(y_test, rf.predict(X_te)))
print("Frequencies:", acc_freqs)

plt.figure(figsize=(6, 4))
plt.plot(list(range(1, 10)), acc_freqs, marker="o")
plt.xlabel("Numero de frecuencias DCT")
plt.ylabel("Exactitud")
plt.grid(True)
plt.tight_layout()
plt.savefig("accuracy_frequencies.jpg")
plt.show()

In [ ]:
acc_quals = []
for regex in qualities_regex:
    X_tr = X_train_all.filter(regex=regex)
    X_te = X_test_all.filter(regex=regex)
    rf.fit(X_tr, y_train)
    acc_quals.append(accuracy_score(y_test, rf.predict(X_te)))
print("Qualities:", acc_quals)

plt.figure(figsize=(6, 4))
plt.plot([80, 85, 90, 95, 100], acc_quals, marker="o")
plt.xlabel("Calidad JPEG maxima incluida")
plt.ylabel("Exactitud")
plt.grid(True)
plt.tight_layout()
plt.savefig("accuracy_qualities.jpg")
plt.show()

In [ ]:
acc_divs = []
for regex in divergences_regex:
    X_tr = X_train_all.filter(regex=regex)
    X_te = X_test_all.filter(regex=regex)
    rf.fit(X_tr, y_train)
    acc_divs.append(accuracy_score(y_test, rf.predict(X_te)))
print("Divergences:", acc_divs)

plt.figure(figsize=(6, 4))
plt.plot(["JS", "JS+Renyi", "JS+Renyi+Tsallis"], acc_divs, marker="o")
plt.xlabel("Divergencias incluidas")
plt.ylabel("Exactitud")
plt.grid(True)
plt.tight_layout()
plt.savefig("accuracy_divergences.jpg")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

axs[0,0].plot([1,2,3,4], acc_bases, marker="o")
axs[0,0].set_xlabel("Numero de bases")
axs[0,0].set_ylabel("Exactitud")
axs[0,0].set_title("Bases")
axs[0,0].grid(True)

axs[0,1].plot(list(range(1,10)), acc_freqs, marker="o")
axs[0,1].set_xlabel("Numero de frecuencias DCT")
axs[0,1].set_ylabel("Exactitud")
axs[0,1].set_title("Frecuencias DCT")
axs[0,1].grid(True)

axs[1,0].plot([80,85,90,95,100], acc_quals, marker="o")
axs[1,0].set_xlabel("Calidad JPEG maxima incluida")
axs[1,0].set_ylabel("Exactitud")
axs[1,0].set_title("Calidades de cuantizacion")
axs[1,0].grid(True)

axs[1,1].plot(["JS", "JS+R", "JS+R+T"], acc_divs, marker="o")
axs[1,1].set_xlabel("Divergencias incluidas")
axs[1,1].set_ylabel("Exactitud")
axs[1,1].set_title("Divergencias")
axs[1,1].grid(True)

plt.tight_layout()
plt.savefig("accuracy_comparison.jpg")
plt.show()
